# 日志与监控（Logging and Monitoring）

假设自动化交易策略已部署到远程服务器，所需的 Python 软件包已经安装，JupyterLab 也已配置好安全访问。如果交易者不想一直登录服务器、守在屏幕前，应如何了解策略的运行情况？原书从日志记录和实时监控两方面回答这个问题。

本 notebook 参考原书的 “Automated Trading Strategy” 和 “Strategy Monitoring” 两份脚本，延续前文的天勤交易背景，按以下顺序展开：

1. **最小示例**：使用预设消息，在同一进程中验证日志写入、发布和接收，理解基本机制。
2. **接入交易策略**：将这套机制放进在线交易流程，自动记录模型输入、预测和交易事件，明确何时记录、记录什么。
3. **独立监控端**：将接收端放到另一个进程或另一台电脑上，查看交易程序发布的消息。

第一部分的本地示例已经运行，实际输出保存在代码下方；后两部分提供完整代码，实时交易和远程监控尚未启动。


## 1. 最小示例：写日志、发布与接收消息

本节只验证一条预设消息如何同时写入文件并传给接收端。发布端和订阅端在同一进程中运行，不调用交易模型。

### 1.1. 日志与监控的用途

**日志记录（Logging）** 将运行信息和事件写入磁盘，供事后检查和分析。这是软件开发与部署中的常规做法。在算法交易中，除了程序是否正常运行，还应关注具有金融含义的信息，例如模型输入、预测方向、持仓和交易结果。

**实时监控（Real-time Monitoring）** 则让交易者及时看到策略正在做什么。原书通过 socket 通信，将重要信息持续发送出来，使本地电脑也能接收和处理远程服务器上策略的运行消息。

| 方式 | 信息去向 | 主要用途 |
|---|---|---|
| 日志记录 | 服务器上的文件等存储介质 | 事后排查、复盘与分析 |
| 实时监控 | 通过 socket 发给监控程序 | 及时查看策略运行情况 |

同一个事件可以既写入日志，又发送到监控端。日志与监控因此可以共用消息内容，但两者用途不同：日志用于留存，监控用于即时观察。

### 1.2. 写入日志并发布消息

原书通过自定义函数，将策略运行中的重要信息写入文本日志，同时通过 ZeroMQ 发送给监控程序。这样，交易者既可以在事后查看日志、复盘策略行为，也可以在本地电脑上实时观察远程服务器中的策略运行情况，无需始终登录服务器、守在交易程序前。

记录的内容不仅包括程序是否正常运行，还可以包括已处理的行情数量、最近的模型输入、预测信号、当前持仓以及交易结果。这些信息有助于还原每一轮交易判断的过程，分析策略为什么下单、为什么保持持仓，以及实际执行是否符合预期。

ZeroMQ 的 Python 接口由 pyzmq 软件包提供。下面沿用原书的思路，用 Python 的 logging 模块保存文本日志，并通过 ZeroMQ 的“发布／订阅”模式传递同一条消息：策略端负责发布，监控端负责接收和展示。

为便于在一个 notebook 中观察完整过程，本示例将两个通信端放在同一进程，通过 inproc:// 地址通信。实际远程部署时，两端分别运行在服务器和本地电脑上，通过网络连接。本示例先使用预设的持仓与信号，集中演示日志记录和消息收发；第 2 节再将这一机制接入交易模型与天勤模拟交易流程。

In [1]:
import logging
from datetime import datetime
from pathlib import Path
import zmq

'''
logger：接收日志消息。
handler：决定写到哪里，例如文件或终端。
formatter：决定输出格式。
PUB / SUB：一端发布，一端订阅。
bind / connect：两端通过对应地址建立通信。
'''

log_path = Path('logging_demo.log')                        # 指定日志文件路径，位于当前工作目录
logger = logging.getLogger('trading_demo')                 # 获取名为 trading_demo 的日志记录器，没有就创建一个
logger.setLevel(logging.INFO)                              # 仅处理 INFO 及以上级别的日志，例如 WARNING 和 ERROR
logger.propagate = False                                   # 不将日志继续传给上级记录器，避免上级再次输出同一条消息

handler = logging.FileHandler(log_path, encoding='utf-8')  # 创建文件处理器，以 UTF-8 编码追加写入；文件不存在时创建
handler.setFormatter(logging.Formatter('%(message)s'))     # 只输出消息正文，不额外添加时间或日志级别
logger.addHandler(handler)                                 # 将文件处理器接到记录器上，使 logger.info() 等调用写入该文件

context = zmq.Context()                                    # 创建 ZeroMQ 上下文，管理后续 socket 所需的通信资源
publisher = context.socket(zmq.PUB)                        # 创建发布端 socket，用于向订阅端发送消息
publisher.bind('inproc://trading_demo')                    # 让发布端绑定到进程内地址 trading_demo，供同一上下文的订阅端连接
subscriber = context.socket(zmq.SUB)                       # 创建订阅端 socket，用于接收发布端的消息
subscriber.setsockopt_string(zmq.SUBSCRIBE, '')            # 设置订阅的消息前缀；空字符串匹配全部消息
subscriber.connect('inproc://trading_demo')                # 将订阅端连接到发布端绑定的同一进程内地址


def log_and_publish(message):
    record = f'{datetime.now().astimezone().isoformat(timespec="seconds")} | {message}'
    logger.info(record)
    publisher.send_string(record)
    return record


In [2]:
# 演示已有目标方向与新信号相同的情况，未执行真实预测或交易。
position, signal = 1, 1
message = (
    f'instrument=KQ.m@SHFE.rb | position={position:+d} | signal={signal:+d} | '
    'NO TRADE PLACED | END OF CYCLE'
)
sent = log_and_publish(message)
print('日志文件：', log_path.resolve())
print('已写入日志并发布消息：')
print(sent)


日志文件： /home/jinting/Projects/kb-quant-finance/attachments/Python-算法交易学习笔记-Cpv4djLFwoWd2CxiXgsc4laSn2f/code/logging_demo.log
已写入日志并发布消息：
2026-11-01T15:06:16+08:00 | instrument=KQ.m@SHFE.rb | position=+1 | signal=+1 | NO TRADE PLACED | END OF CYCLE


本例中，position = 1 表示当前为多头，signal = 1 表示本轮预测仍为上涨，两者方向一致，因此无需换向，输出 NO TRADE PLACED。随后输出 END OF CYCLE，表示本轮处理结束。同一条消息既被写入日志文件，也通过 ZeroMQ 发布给监控端。这里的持仓与信号均为预设的演示值。

### 1.3. 接收消息并核对日志

订阅端接收发布端发送的消息。实际远程部署时，策略进程与监控进程分别运行在服务器和本地电脑，通过网络连接；本地示例让两端共用一个 ZeroMQ context，便于直接观察相同的信息。

下面用一秒的接收等待上限完成演示，避免没有消息时 notebook 一直阻塞。

In [3]:
if not subscriber.poll(timeout=1000):  # 毫秒；最多等待 1 秒
    raise TimeoutError('监控端未在 1 秒内收到消息。')
received = subscriber.recv_string()
print('监控端收到：')
print(received)
print('与发布内容一致：', received == sent)


监控端收到：
2026-11-01T15:06:16+08:00 | instrument=KQ.m@SHFE.rb | position=+1 | signal=+1 | NO TRADE PLACED | END OF CYCLE
与发布内容一致： True


In [4]:
# 读取日志文件，核对同一条消息是否已保存。
last_record = log_path.read_text(encoding='utf-8').splitlines()[-1]
print('日志文件：', log_path.resolve())
print('最后一条记录：')
print(last_record)
print('与监控端收到的内容一致：', last_record == received)


日志文件： /home/jinting/Projects/kb-quant-finance/attachments/Python-算法交易学习笔记-Cpv4djLFwoWd2CxiXgsc4laSn2f/code/logging_demo.log
最后一条记录：
2026-11-01T15:06:16+08:00 | instrument=KQ.m@SHFE.rb | position=+1 | signal=+1 | NO TRADE PLACED | END OF CYCLE
与监控端收到的内容一致： True


监控端接收到了发布的消息，日志文件也保存了相同内容。这样，一条交易运行事件便同时具备了即时观察和事后查阅两种用途。

本示例只验证本地消息收发。ZeroMQ 的 PUB/SUB 消息不会为未连接的订阅端补发历史记录，因此监控端断开期间的信息仍应从日志中查阅。原书也指出，文本日志可以按需要改成数据库或 HDF5 等存储方式，相关内容可参见第 3 章。

In [5]:
# 演示完成后关闭文件和通信端。
logger.removeHandler(handler)
handler.close()
subscriber.close(linger=0)
publisher.close(linger=0)
context.term()
print('日志文件和 ZeroMQ 通信端已关闭。')


日志文件和 ZeroMQ 通信端已关闭。


## 2. 接入交易策略：消息发送

上一节通过预设消息演示了日志写入和消息收发。本节将这些功能加入在线交易程序，使策略在接收行情、生成预测和调整持仓的同时，自动记录并发送运行信息。

参考原书，每完成一轮预测，程序会记录已处理的行情数量、最近的模型输入、当前目标方向和新信号。方向发生变化时，记录新的目标持仓；方向不变时，记录“无需交易”。天勤随后返回的实际成交，以及程序结束时的平仓过程，也会单独记录。这些信息既保存在日志文件中，供事后检查，也发送给监控端，便于实时观察。

代码沿用前一节的天勤在线交易逻辑，从 trading_common.py 导入预测类和账户连接函数，再通过 logger_monitor() 统一写入日志、发布消息，通过 report_trade() 打印和记录交易事件。模型及特征参数保持与离线训练一致。

### 2.1. 日志与发布

原书每次启动重新创建 `automated_strategy.log`，随后将消息追加写入，并通过 ZeroMQ 发布。`time` 控制是否加时间，`sep` 控制是否加分隔线。

In [6]:
# 交易接口替换为天勤，日志与 ZeroMQ 发布/订阅流程沿用原书。
import datetime as dt
import pickle

import numpy as np
import pandas as pd
import zmq


def logger_monitor(message, publisher, log_file='automated_strategy.log', time=True, sep=True):
    """同一条消息写入日志文件，并通过 ZeroMQ 发布。"""
    msg = '\n' + str(dt.datetime.now()) + '\n' if time else ''
    if sep:
        msg += 80 * '=' + '\n'
    msg += str(message) + '\n\n'
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(msg)
    publisher.send_string(msg)


### 2.2. 加载前一节的预测逻辑

导入 'MLTrader' ，下一步通过继承扩展日志行为，不改变特征和预测方式。

In [7]:
from trading_common import MLTrader


### 2.3. 记录模型输入、信号和交易事件

下面的 `MonitoredMLTrader` 继承公共模块中的 `MLTrader`。它重写 `on_bar()`，在原有的流程中加入日志记录，并整理的原书的 `report_trade()`，统一打印、记录和发布交易事件。原书将这些功能集中在 `MLTrader` 中。

`report_trade()` 负责报告事件，本身不下单。尤其要区分目标与成交：例如模型希望从多一手变为空一手，日志会先记录新的空头目标，但天勤仍需执行平多、开空。`GOING SHORT (TARGET)` 因此只表示模型产生了空头目标，不能说明账户已经持有空头。

天勤的目标持仓任务异步执行(设置目标后，程序不等成交完成，就会继续往下执行)。实际成交信息要在后续账户更新中读取。下一节的行情循环收到成交后，会再次调用 `report_trade()`，以 `EXECUTED` 标记实际成交。将目标方向、实际多空持仓和成交事件分开记录，才能核对策略希望执行的操作是否已经完成。

程序持续统计报价更新次数，已处理的 K 线数。每完成一次预测，将截至当时的累计次数和 K 线数以及最近最多五次预测的标准化输入写入日志。

`features` 是本轮模型实际使用的一行输入.

`previous target direction` 是上一轮的目标方向，`signal` 是本轮预测方向。比较这两个方向，可以判断是否需要改变目标持仓：方向变化时记录 `GOING LONG/SHORT (TARGET)`，相同时记录 `NO TRADE PLACED`，随后返回目标手数，交给下一节的行情循环设置持仓。

In [8]:
class MonitoredMLTrader(MLTrader):
    def __init__(self, algorithm, publisher, log_file, volume=1):
        super().__init__(algorithm, volume)
        self.publisher = publisher
        self.log_file = log_file
        self.recent_inputs = []
        self.bars = 0

    def report_trade(self, event, details):
        """打印、记录并发送目标指令或实际成交；通过 event 区分。"""
        message = f'*** {event} ***\n{details}'
        logger_monitor(message, publisher=self.publisher, log_file=self.log_file)
        print(message)

    def on_bar(self, close, updates=0):
        x = self.prepare_features(close)
        signal = int(self.model.predict(x)[0])
        self.bars += 1
        x.index = [close.index[-1]]                                                          # 本行输入所使用的最后一根已结束 K 线
        self.recent_inputs.append(x)
        self.recent_inputs = self.recent_inputs[-5:]

        logger_monitor(f'QUOTE UPDATES: {updates} | PROCESSED BARS: {self.bars}', publisher=self.publisher, log_file=self.log_file)
        logger_monitor('MOST RECENT INPUTS\n' + pd.concat(self.recent_inputs).to_string(), time=False, publisher=self.publisher, log_file=self.log_file)
        logger_monitor(f'features:\n{x.to_numpy()}\n'
                 f'previous target direction: {self.position}\n'
                 f'signal: {signal}', time=False, publisher=self.publisher, log_file=self.log_file)
        if signal != self.position:
            direction = 'LONG' if signal == 1 else 'SHORT'
            self.report_trade(f'GOING {direction} (TARGET)',
                              {'target_volume': signal * self.volume})
        else:
            logger_monitor('*** NO TRADE PLACED ***', publisher=self.publisher, log_file=self.log_file)
        self.position = signal
        logger_monitor('*** END OF CYCLE ***', time=False, sep=False, publisher=self.publisher, log_file=self.log_file)
        return signal * self.volume


### 2.4. 消息发送

下面接收行情，调用带日志的预测方法，并记录天勤返回的实际成交。等待上限沿用前一节的设置，避免休市时持续阻塞。结束时将目标持仓设为零，并记录平仓过程。
`make_api()` 从 `trading_common.py` 导入，与前一节共用天勤模拟连接配置。


In [9]:
# 实时交易需要天勤 SDK；CSV 验证只需公共模块中的 MLTrader。
import time
from tqsdk import TargetPosTask
from trading_common import make_api


在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [10]:
def report_fills(api, trader, seen_trades):
    """读取新成交并报告；原地更新传入的 seen_trades，避免重复报告。"""
    trades = api.get_trade()
    for trade_id, trade in trades.items():
        if trade_id not in seen_trades:
            trader.report_trade('EXECUTED', dict(trade))
            seen_trades.add(trade_id)


def run_monitored(algorithm, stop=150, timeout=60,
                  bind_address='tcp://127.0.0.1:5555',
                  log_file='automated_strategy.log'):
    
    # 1. 创建日志文件和发布端。与原书一致，每次启动重新建立日志。
    with zmq.Context() as context, context.socket(zmq.PUB) as publisher:
        publisher.setsockopt(zmq.LINGER, 0)
        publisher.bind(bind_address)
        with open(log_file, 'w', encoding='utf-8') as f:
            f.write('*** NEW LOG FILE ***\n' + str(dt.datetime.now()) + '\n')
        trader = MonitoredMLTrader(algorithm, publisher, log_file)

        # 2. 连接天勤模拟账户，准备行情及实际合约的目标持仓任务。
        with make_api() as api:
            quote = api.get_quote(trader.instrument)
            klines = api.get_kline_serial(trader.instrument, 
                                          trader.bar_seconds, 
                                          data_length=max(100, trader.window + trader.lags + 1)
                                         )
            deadline = time.time() + timeout
            while not (api.is_serial_ready(klines) and quote.underlying_symbol):
                if not api.wait_update(deadline=deadline):
                    raise TimeoutError('行情初始化超时。')
            contract = quote.underlying_symbol
            target_task = TargetPosTask(api, contract)
            position = api.get_position(contract)
            logger_monitor(f'SIGNAL INSTRUMENT: {trader.instrument} | CONTRACT: {contract}', publisher=publisher, log_file=log_file)
            
            # 不同交易所、账户或系统可能使用相同的编号，只要在各自规定的范围内能区分即可。            
            # 实际成交由天勤异步返回，独立于前面的目标持仓指令。
            seen_trades = set(api.get_trade())  # 跳过启动时已有成交；传参不会复制此集合。
            # 3. 新 K 线结束时记录输入与信号，再设置目标持仓。
            updates = 0
            deadline = time.time() + timeout
            try:
                while updates < stop:
                    if not (api.wait_update(deadline=deadline) and time.time() < deadline):
                        logger_monitor(f'连续 {timeout} 秒无报价更新，结束运行。', publisher=publisher, log_file=log_file)
                        break
                    report_fills(api, trader, seen_trades)
                    if api.is_changing(quote):
                        updates += 1
                        deadline = time.time() + timeout # 有了新的行情，更新deadline，避免空等
                    if api.is_changing(klines.iloc[-1], 'datetime'):
                        target = trader.on_bar(klines.iloc[:-1]['close'], updates)
                        target_task.set_target_volume(target)
                        logger_monitor(f'ACTUAL POSITION: long={position.pos_long}, short={position.pos_short}', publisher=publisher, log_file=log_file)

            # 4. 结束时平仓，记录平仓指令与随后收到的实际成交。
            finally:
                target_task.set_target_volume(0)
                trader.report_trade('GOING NEUTRAL (TARGET)', {'target_volume': 0})
                deadline = time.time() + 30
                while api.wait_update(deadline=deadline):
                    report_fills(api, trader, seen_trades)
                    if position.pos_long == 0 and position.pos_short == 0:
                        trader.position = 0
                        logger_monitor('当前持仓已清空。', publisher=publisher, log_file=log_file)
                        break
                else:
                    logger_monitor(f'平仓确认超时：多 {position.pos_long} 手，空 {position.pos_short} 手。', publisher=publisher, log_file=log_file)


In [11]:
# 与前一节使用同一个模型文件。
with open('algorithm.pkl', 'rb') as f:
    algorithm = pickle.load(f)

# 手动取消注释才会连接天勤并提交模拟委托。
run_monitored(algorithm, stop=150, timeout=60)


*** GOING NEUTRAL (TARGET) ***
{'target_volume': 0}


## 3. 接入交易策略：消息接收

上一节的交易程序会把运行信息写进日志，同时通过网络发送出去。这一节另外写一个接收程序，把收到的消息显示出来。这样，交易程序可以放在服务器上运行，我们在自己的电脑上就能看到它产生了什么信号、准备买卖多少手，以及实际成交了多少。

原书是这样做的：先创建一个接收消息的 `SUB` socket，连接交易程序的地址，再设置接收全部消息。接着用 `while True` 一直循环，`recv_string()` 收到一条消息，就用 `print()` 打印一条；没有消息，就停在那里等。

下面把这些步骤写进 `monitor_strategy()`。调用时，`address` 指定去哪里接收消息，`timeout` 指定最多等多久。这里默认连续60秒没有消息就结束。结束的只是这个查看消息的程序，交易程序仍然自己运行，不会因此平仓。这里等待的是交易程序发来的消息，不是上一节的新行情。

在同一台电脑上测试时，两边都使用默认地址 `tcp://127.0.0.1:5555`，需要开两个独立的 Python 进程。或者两个使用**不同内核**的 notebook：一个运行 `monitor_strategy()`，另一个运行 `run_monitored()`。

如果交易程序放在另一台服务器上，就要把 `address` 中的 `127.0.0.1` 换成服务器的实际地址，因为 `127.0.0.1` 只代表当前这台电脑。服务器上的交易程序也要允许外部连接：原书用 `tcp://0.0.0.0:5555` 接收连接，意思是允许通过本机各个网卡连接这个端口。自己的电脑连接时仍然填写服务器实际地址，并确认服务器允许访问5555端口。

这个接收程序只能看到连接建立以后发来的新消息，不会补收之前的消息。所以最好先启动它，但连接也需要一点时间，最早的消息仍有可能错过。漏看的内容可以去交易程序的日志文件里找。


In [12]:
def monitor_strategy(address='tcp://127.0.0.1:5555', timeout=60):
    # 订阅端在另一个 notebook 内核或 Python 进程中运行。
    with zmq.Context() as context, context.socket(zmq.SUB) as subscriber:
        subscriber.setsockopt_string(zmq.SUBSCRIBE, '')  # 接收全部消息
        subscriber.connect(address)
        while subscriber.poll(timeout=int(timeout * 1000)):
            print(subscriber.recv_string())
        print(f'连续 {timeout} 秒未收到监控消息，结束接收。')


# 手动运行；远程监控时，将地址改为发布端可达的服务器地址。
monitor_strategy('tcp://127.0.0.1:5555', timeout=60)


连续 60 秒未收到监控消息，结束接收。


## 4. 补充

- 日志和监控消息可按需调整。
- 示例未实现止损、止盈，需另行添加。
- 实盘前需检查交易与执行逻辑，并处理通信异常、行情延迟、tick 丢失等风险。
- 本地测试通过，不代表远程运行稳定或策略能够盈利。